# Leave-One-Out
метод кроссвалидации. Крайний случай K-Fold, когда размер каждого фолда равен 1, тогда тестовая выборка для каждой модели состоит из одного сэмпла:

* Полезно при малом количестве данных, когда мы хотим использовать максимальное их количество для обучения, но количество итераций равно количеству сэмплов, поэтому только для малого количества данных

# Grid Search:


1.   Фиксируем заранее сетку значений параметров
2.   обучаем и тестируем модель на всех комбинациях значений гиперпараметров
3.   выбираем лучшую комбинацию


#Random Search:


1.   для каждого параметра задаётся распределение, из которого выбирается его значение
2.   Комбинация составляется сэмплированием распределений
3.   модели обуччаются и тестируются
4.   выбирается лучшая комбинация

Намного быстрее grid search, т.к. смотрит не все комбинации, но может найти не абсолютный оптимум



#Байесовская оптимизация:

состоит из:
1. вероятностной модели, которая приближает распределение параметра, исходя из исторических данных
2. acquisition function, которая указывает по статистикам первой, где вычислять следующее значение

В общем случае:
на t+1-итерации вычисляется $x_{t+1}$, максимизирующий $α(x|S_t)$, $S_t$ - множество предыдущих значений
вычисляется acquisition function и обновляется $S_t$
обновляется статистическая модель

# Классификация методов отбора признаков


1.   Unsupervise (не используется таргет)
2.   Supervised (используется таргет)

* Wrappers - оборачивают модель, множество раз обучая её и оценивая текущую комбинацию признаков (как Grid Search)
* Filters - не используют модель вообще, отбрасывая признаки, исходя из статистических связей в данных
* Embedded - отбор во время обучения модели

* Коэффициент корреляции Пирсона: ближе к 1 или -1 - полезнее, к 0 - выбрасываем. Работает для непрерывных величин с нормальным распределением, поэтому для категориальных признаков не применим.

* Lasso зануляет признаки, которые слабо воздействуют на таргет. Это условная оптимизация: задача минимизации функции потерь без регуляризационного члена с ограничением, где $||w||_1 <= c$. У $L_1$-нормы единичная сфера - многомерный октаэдр, т.е., ромб. Тогда задача - найти оптимум внутри ромба или на его границе. Если построить линии уровня функции потерь и этот октаэдр на одном графике, можно увидеть, что точка касания линии уровня функции потерь с большей вероятностью будет лежать в углах октаэдра, а его углы находятся там, где часть координат зануляется. То есть, модель сама занулила некоторые веса в процессе обучения.

* Permutation significance. Значения признака перемешиваются между объектами, затем считается метрика на получившейся модели и смотрится, насколько ухудшились метрики. Если сильно, то признак важен. Не работает с линейно зависимыми признаками, т.к. один заменяет другой

* SHAP - самый современный метод отбора признаков. Учитывает влияние признака на предсказание, используя теорию игр.

* Критерий Хи-квадрат. Статистический критерий, позволяющий понять, есть ли значимая зависимость между двумя категориальными случайными величинами. Используется в задаче классификации, позволяя понять, какие признаки связаны с таргетом

In [1]:
import pandas as pd

In [2]:
data = pd.read_json('/content/drive/MyDrive/ML/datasets ML1-3/train.json')

In [3]:
data['interest_level'] = data['interest_level'].map({'low': 0, 'medium': 1, 'hight': 2})

In [4]:
features = ['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace']

data['features'] = data['features'].map(lambda x: [f.replace(' ', '').replace('"', '').replace("'", '') for f in x])

for f in features:
  data[f] = data['features'].apply(lambda x: int(f in x))

features.extend(['bathrooms', 'bedrooms'])

In [5]:
X, y = data[features], data['price']

In [ ]:
import numpy as np

def my_train_test_split(X, y, test_size=0.2, shuffle=True, random_state=21):
    indices = np.arange(len(X))

    if shuffle:
      rng = np.random.RandomState(random_state)
      rng.shuffle(indices)

    train_n = int(len(X) * (1 - test_size))

    train_idx = indices[:train_n]
    test_idx = indices[train_n:]

    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

In [11]:
def my_train_val_test_split(X, y, test_size=0.2, val_size=0.1, shuffle=True, random_state=21):
    indices = np.arange(len(X))

    if shuffle:
      rng = np.random.RandomState(random_state)
      rng.shuffle(indices)

    train_n = int(len(X) * (1 - (test_size + val_size)))
    val_n = int(len(X) * val_size)

    train_idx = indices[:train_n]
    val_idx = indices[train_n: train_n + val_n]
    test_idx = indices[train_n + val_n:
                     ]
    return X.iloc[train_idx], X.iloc[val_idx],  X.iloc[test_idx], y.iloc[train_idx], y.iloc[val_idx], y.iloc[test_idx]

In [ ]:
def my_date_split(X, y, date_split):
    X = X.copy()

    X['created'] = pd.to_datetime(X['created'])
    date_split = pd.to_datetime(date_split)

    train_mask = X['created'] < date_split
    test_mask = X['created'] >= date_split

    X_train = X[train_mask]
    X_test = X[test_mask]

    y_train = y[train_mask]
    y_test = y[test_mask]

    return X_train, X_test, y_train, y_test

In [ ]:
def my_date_val_split(X, y, val_date, test_date):
    X = X.copy()

    X['created'] = pd.to_datetime(X['created'])
    val_dt = pd.to_datetime(val_date)
    test_dt = pd.to_datetime(test_date)

    train_mask = X['created'] < val_dt
    val_mask = (X['created'] >= val_dt) & (X['created'] < test_dt)
    test_mask = X['created'] >= test_dt

    X_train = X[train_mask]
    X_val = X[val_mask]
    X_test = X[test_mask]

    y_train = y[train_mask]
    y_val = y[val_mask]
    y_test = y[test_mask]

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
def my_kfold_split(X, k=5, shuffle=True, random_state=21):
    n = len(X)
    indices = np.arange(n)

    if shuffle:
        rng = np.random.RandomState(random_state)
        rng.shuffle(indices)

    fold_sizes = np.full(k, n // k)
    fold_sizes[:n % k] += 1

    folds = []
    current = 0

    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        test_idx = indices[start:stop]

        train_idx = np.concatenate([indices[:start], indices[stop:]])

        folds.append((train_idx, test_idx))

        current = stop

    return folds

In [ ]:
import numpy as np

def my_group_kfold_split(X, k=5, group_field='building_id', shuffle=True, random_state=21):
    groups = X[group_field].values
    unique_groups = np.unique(groups)

    if shuffle:
        rng = np.random.RandomState(random_state)
        rng.shuffle(unique_groups)

    n_groups = len(unique_groups)

    fold_sizes = np.full(k, n_groups // k)
    fold_sizes[:n_groups % k] += 1

    folds = []
    current = 0

    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        test_groups = unique_groups[start:stop]

        test_mask = np.isin(groups, test_groups)
        train_mask = ~test_mask

        test_idx = np.where(test_mask)[0]
        train_idx = np.where(train_mask)[0]

        folds.append((train_idx, test_idx))

        current = stop

    return folds

In [ ]:
import numpy as np

def my_stratified_kfold_split(X, y, k=5, shuffle=True, random_state=21):
    n = len(X)
    indices = np.arange(n)

    if shuffle:
        rng = np.random.RandomState(random_state)

    unique_classes = np.unique(y)
    class_indices = {}

    for cls in unique_classes:
        cls_idx = indices[y == cls]
        if shuffle:
          rng.shuffle(cls_idx)
        class_indices[cls] = cls_idx

    folds = [[] for _ in range(k)]

    for cls in unique_classes:
        cls_idx = class_indices[cls]

        fold_sizes = np.full(k, len(cls_idx) // k)
        fold_sizes[:len(cls_idx) % k] += 1

        current = 0
        for i in range(k):
            start, stop = current, current + fold_sizes[i]
            folds[i].extend(cls_idx[start:stop])
            current = stop

    result = []
    for i in range(k):
        test_idx = np.array(folds[i])
        train_idx = np.setdiff1d(indices, test_idx)

        result.append((train_idx, test_idx))

    return result

In [ ]:
def my_time_series_split(X, k=5, date_field='created'):
    X_sorted = X.sort_values(by=date_field)
    indices = X_sorted.index.to_numpy()

    n = len(X)
    fold_size = n // (k + 1)
    splits = []

    for i in range(1, k + 1):
        train_end = fold_size * i
        test_end = fold_size * (i + 1)

        train_idx = indices[:train_end]
        test_idx = indices[train_end:test_end]

        splits.append((train_idx, test_idx))

    return splits

In [ ]:
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    GroupKFold,
    TimeSeriesSplit
)

train_test_split

In [ ]:
X_train, X_test, y_train, y_test = my_train_test_split(X, y)

print(f"Длина исходного сета: {len(data)};\nДлина обучающей выборки: {len(X_train)};\nДлина тестовой выборки: {len(X_test)};\n.")

Длина исходного сета: 49352;
Длина обучающей выборки: 39481;
Длина тестовой выборки: 9871;
.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, random_state=21, test_size=0.2)

print(f"Длина исходного сета: {len(data)};\nДлина обучающей выборки: {len(X_train)};\nДлина тестовой выборки: {len(X_test)};\n")

Длина исходного сета: 49352;
Длина обучающей выборки: 39481;
Длина тестовой выборки: 9871;



KFold

In [ ]:
k = 5

kf_skl = KFold(n_splits=k, shuffle=True, random_state=21)
kf_my = my_kfold_split(X, k=k, shuffle=True, random_state=21)

for i, ((train_idx_my, test_idx_my), (train_idx_skl, test_idx_skl)) in enumerate(
    zip(kf_my, kf_skl.split(X))
):
    print(f"\nFold {i+1}:")
    print(f"My train: {len(train_idx_my)}, test: {len(test_idx_my)}")
    print(f"Sk train: {len(train_idx_skl)}, test: {len(test_idx_skl)}")


import numpy as np

print("\n=== Сравнение средних значений признаков ===")

for i, ((train_idx_my, _), (train_idx_skl, _)) in enumerate(
    zip(kf_my, kf_skl.split(X))
):
    X_train_my = X.iloc[train_idx_my]
    X_train_skl = X.iloc[train_idx_skl]

    diff = (X_train_my.mean() - X_train_skl.mean()).abs().mean()

    print(f"Fold {i+1}: mean diff = {diff:.6f}")


Fold 1:
My train: 39481, test: 9871
Sk train: 39481, test: 9871

Fold 2:
My train: 39481, test: 9871
Sk train: 39481, test: 9871

Fold 3:
My train: 39482, test: 9870
Sk train: 39482, test: 9870

Fold 4:
My train: 39482, test: 9870
Sk train: 39482, test: 9870

Fold 5:
My train: 39482, test: 9870
Sk train: 39482, test: 9870

=== Сравнение средних значений признаков ===
Fold 1: mean diff = 0.000000
Fold 2: mean diff = 0.000000
Fold 3: mean diff = 0.000000
Fold 4: mean diff = 0.000000
Fold 5: mean diff = 0.000000


In [ ]:
from sklearn.model_selection import GroupKFold

k = 5
X_group = X.copy()
X_group['building_id'] = data['building_id']
groups = X_group['building_id']

gkf_skl = GroupKFold(n_splits=k)
gkf_my = my_group_kfold_split(X_group, k=k, group_field='building_id')

print("=== Размеры фолдов ===")

for i, ((train_idx_my, test_idx_my), (train_idx_skl, test_idx_skl)) in enumerate(
    zip(gkf_my, gkf_skl.split(X_group, y, groups))
):
    print(f"\nFold {i+1}:")
    print(f"My      -> train: {len(train_idx_my)}, test: {len(test_idx_my)}")
    print(f"Sklearn -> train: {len(train_idx_skl)}, test: {len(test_idx_skl)}")

=== Размеры фолдов ===

Fold 1:
My      -> train: 33053, test: 16299
Sklearn -> train: 39481, test: 9871

Fold 2:
My      -> train: 41191, test: 8161
Sklearn -> train: 39481, test: 9871

Fold 3:
My      -> train: 41086, test: 8266
Sklearn -> train: 39482, test: 9870

Fold 4:
My      -> train: 40716, test: 8636
Sklearn -> train: 39482, test: 9870

Fold 5:
My      -> train: 41362, test: 7990
Sklearn -> train: 39482, test: 9870


Версия sklearn учитывает ещё и размер групп, а моя версия только их количество, отсюда различие

In [ ]:
from sklearn.model_selection import StratifiedKFold

k = 5

skf_skl = StratifiedKFold(n_splits=k, shuffle=True, random_state=21)
skf_my = my_stratified_kfold_split(X, y, k=k)

print("=== Размеры фолдов ===")

for i, ((train_idx_my, test_idx_my),
        (train_idx_skl, test_idx_skl)) in enumerate(
    zip(skf_my, skf_skl.split(X, y))
):
    print(f"\nFold {i+1}:")
    print(f"My      -> train: {len(train_idx_my)}, test: {len(test_idx_my)}")
    print(f"Sklearn -> train: {len(train_idx_skl)}, test: {len(test_idx_skl)}")


print("\n=== Распределение классов ===")

for i, ((train_idx_my, test_idx_my),
        (train_idx_skl, test_idx_skl)) in enumerate(
    zip(skf_my, skf_skl.split(X, y))
):
    y_train_my = y.iloc[train_idx_my]
    y_train_skl = y.iloc[train_idx_skl]

    dist_my = y_train_my.value_counts(normalize=True).sort_index()
    dist_skl = y_train_skl.value_counts(normalize=True).sort_index()

    diff = (dist_my - dist_skl).abs().mean()

    print(f"Fold {i+1}: class distribution diff = {diff:.6f}")

=== Размеры фолдов ===

Fold 1:
My      -> train: 37830, test: 11522
Sklearn -> train: 39481, test: 9871

Fold 2:
My      -> train: 39159, test: 10193
Sklearn -> train: 39481, test: 9871

Fold 3:
My      -> train: 39818, test: 9534
Sklearn -> train: 39482, test: 9870

Fold 4:
My      -> train: 40177, test: 9175
Sklearn -> train: 39482, test: 9870

Fold 5:
My      -> train: 40424, test: 8928
Sklearn -> train: 39482, test: 9870

=== Распределение классов ===


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fold 1: class distribution diff = 0.000031
Fold 2: class distribution diff = 0.000008
Fold 3: class distribution diff = 0.000008
Fold 4: class distribution diff = 0.000011
Fold 5: class distribution diff = 0.000013


результаты схожи; у sklearn стабильнее размеры фолдов из-за оптимизации в случае маленьких классов

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

ts_skl = TimeSeriesSplit(n_splits=5)

print("\n=== Проверка временного порядка ===")

X_sorted = X_date.sort_values('created')

for i, (train_idx, test_idx) in enumerate(ts_skl.split(X_sorted)):
    max_train_time = X_sorted.iloc[train_idx]['created'].max()
    min_test_time = X_sorted.iloc[test_idx]['created'].min()

    print(f"Fold {i+1}: train_max < test_min = {max_train_time <= min_test_time}")


=== Проверка временного порядка ===
Fold 1: train_max < test_min = True
Fold 2: train_max < test_min = True
Fold 3: train_max < test_min = True
Fold 4: train_max < test_min = True
Fold 5: train_max < test_min = True

=== Сдвиг распределений ===
Fold 1: feature drift = 0.006339
Fold 2: feature drift = 0.007905
Fold 3: feature drift = 0.006083
Fold 4: feature drift = 0.018970
Fold 5: feature drift = 0.008088


в трейне объекты раньше по дате, чем в тесте

Feature selection

In [ ]:
import numpy as np

X_train, X_val, X_test, y_train, y_val, y_test = my_train_val_test_split(
    X, y,
    test_size=0.2,
    val_size=0.2
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
val_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import log_loss

lasso = Lasso(random_state=21, alpha=0.001)
lasso.fit(train_scaled, y_train)

Lasso(alpha=0.001, random_state=21)

In [ ]:
from sklearn.metrics import root_mean_squared_error

val_pred = lasso.predict(val_scaled)
test_pred = lasso.predict(test_scaled)

print("Validation RMSE:", root_mean_squared_error(y_val, val_pred))
print("Test RMSE:", root_mean_squared_error(y_test, test_pred))

Validation RMSE: 2232.18278358162
Test RMSE: 2008.048901845


In [ ]:
coef = pd.Series(lasso.coef_, index=features)

In [ ]:
sorted_coef = coef.abs().sort_values(ascending=False)
sorted_coef

,0
Doorman,1062.077586
bathrooms,954.297893
bedrooms,771.276310
LaundryinBuilding,403.789679
FitnessCenter,379.521936
Elevator,369.511569
DogsAllowed,364.291712
LaundryInBuilding,273.869148
NoFee,219.243382
LaundryinUnit,215.887867


In [ ]:
top10 = sorted_coef.head(10).index.tolist()

In [ ]:
X_train_10 = X_train[top10]
X_val_10 = X_val[top10]
X_test_10 = X_test[top10]

scaler2 = StandardScaler()

X_train_10_s = scaler2.fit_transform(X_train_10)
X_val_10_s = scaler2.transform(X_val_10)
X_test_10_s = scaler2.transform(X_test_10)

model2 = Lasso(alpha=0.001, random_state=21)
model2.fit(X_train_10_s, y_train)

Lasso(alpha=0.001, random_state=21)

In [ ]:
from sklearn.metrics import root_mean_squared_error

val_pred = model2.predict(X_val_10_s)
test_pred = model2.predict(X_test_10_s)

print("Validation RMSE:", root_mean_squared_error(y_val, val_pred))
print("Test RMSE:", root_mean_squared_error(y_test, test_pred))

Validation RMSE: 2212.360337671813
Test RMSE: 2005.6293764077616


разницы почти нет

In [ ]:
X.apply(lambda col: col.corr(y)).abs()

,0
Elevator,0.023483
HardwoodFloors,0.001525
CatsAllowed,0.011639
DogsAllowed,0.013062
Doorman,0.033936
Dishwasher,0.013157
NoFee,0.005229
LaundryinBuilding,0.006591
FitnessCenter,0.015368
Pre-War,0.002785


In [ ]:
def simple_feature_selection(X, y):
    nan_ratio = X.isna().mean()
    corr = X.apply(lambda col: col.corr(y)).abs()

    score = corr * (1 - nan_ratio)

    return score.sort_values(ascending=False)

In [ ]:
scores = simple_feature_selection(X, y)

print(scores)

bathrooms            0.069661
bedrooms             0.051788
Doorman              0.033936
Elevator             0.023483
LaundryinUnit        0.022481
DiningRoom           0.021228
FitnessCenter        0.015368
Terrace              0.014675
Dishwasher           0.013157
DogsAllowed          0.013062
OutdoorSpace         0.012205
CatsAllowed          0.011639
Balcony              0.011597
SwimmingPool         0.010814
RoofDeck             0.008425
LaundryinBuilding    0.006591
HighSpeedInternet    0.006055
NoFee                0.005229
NewConstruction      0.004571
LaundryInBuilding    0.003385
Pre-War              0.002785
HardwoodFloors       0.001525
dtype: float64


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

top10_features = scores.head(10).index.tolist()

X_train_10 = X_train[top10_features]
X_val_10 = X_val[top10_features]
X_test_10 = X_test[top10_features]

scaler = StandardScaler()

X_train_10_s = scaler.fit_transform(X_train_10)
X_val_10_s = scaler.transform(X_val_10)
X_test_10_s = scaler.transform(X_test_10)

model = Lasso(alpha=0.001, random_state=21)
model.fit(X_train_10_s, y_train)

Lasso(alpha=0.001, random_state=21)

In [ ]:
from sklearn.metrics import root_mean_squared_error

val_pred = model.predict(X_val_10_s)
test_pred = model.predict(X_test_10_s)

print("Validation RMSE:", root_mean_squared_error(y_val, val_pred))
print("Test RMSE:", root_mean_squared_error(y_test, test_pred))

Validation RMSE: 2212.360337671813
Test RMSE: 2005.6293764077616


вышло чуть лучше

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_val_10_s,
    val_pred,
    n_repeats=10,
    random_state=21,
    scoring='neg_mean_squared_error'
)

In [ ]:
result.importances_mean

array([1.88070163e+06, 9.93141400e+05, 2.02649513e+06, 6.22131382e+04,
       6.28727101e+04, 1.91918086e+01, 6.00425478e+05, 1.09456076e-01,
       2.94272053e+05, 2.34841493e+05])

In [ ]:
perm_importance = pd.Series(
    result.importances_mean,
    index=top10
).sort_values(ascending=False)

print(perm_importance)

Doorman          2.026495e+06
bathrooms        1.880702e+06
bedrooms         9.931414e+05
FitnessCenter    6.004255e+05
Dishwasher       2.942721e+05
DogsAllowed      2.348415e+05
LaundryinUnit    6.287271e+04
Elevator         6.221314e+04
DiningRoom       1.919181e+01
Terrace          1.094561e-01
dtype: float64


In [ ]:
top10_perm = perm_importance.head(10).index.tolist()

In [ ]:
X_train_10 = X_train[top10_perm]
X_val_10 = X_val[top10_perm]
X_test_10 = X_test[top10_perm]

scaler = StandardScaler()

X_train_10_s = scaler.fit_transform(X_train_10)
X_val_10_s = scaler.transform(X_val_10)
X_test_10_s = scaler.transform(X_test_10)

model_perm = Lasso(alpha=0.001, random_state=21)
model_perm.fit(X_train_10_s, y_train)

Lasso(alpha=0.001, random_state=21)

In [ ]:
val_pred = model_perm.predict(X_val_10_s)
test_pred = model_perm.predict(X_test_10_s)

print("Permutation RMSE:", root_mean_squared_error(y_val, val_pred))

Permutation RMSE: 2212.3601397232


SHAP

In [ ]:
import shap

explainer = shap.Explainer(model, X_train_10_s)
shap_values = explainer(X_val_10_s)

In [ ]:
import numpy as np

shap_importance = np.abs(shap_values.values).mean(axis=0)

shap_importance = pd.Series(
    shap_importance,
    index=top10
).sort_values(ascending=False)

print(shap_importance)

Doorman          1024.870898
bedrooms          840.336832
bathrooms         512.124626
LaundryinUnit     464.419622
DiningRoom        280.241153
Dishwasher        181.212405
FitnessCenter     166.368458
Terrace           153.001635
DogsAllowed         3.070583
Elevator            0.226192
dtype: float64


In [ ]:
top10_shap = shap_importance.head(10).index.tolist()

In [ ]:
X_train_10 = X_train[top10_shap]
X_val_10 = X_val[top10_shap]
X_test_10 = X_test[top10_shap]

scaler = StandardScaler()

X_train_10_s = scaler.fit_transform(X_train_10)
X_val_10_s = scaler.transform(X_val_10)
X_test_10_s = scaler.transform(X_test_10)

model_shap = Lasso(alpha=0.001, random_state=21)
model_shap.fit(X_train_10_s, y_train)

Lasso(alpha=0.001, random_state=21)

In [ ]:
val_pred = model_shap.predict(X_val_10_s)

print("SHAP RMSE:", root_mean_squared_error(y_val, val_pred))

SHAP RMSE: 2212.360139420091


* SHAP самый долгий - сложные вычисления
* Permutation не быстрый - переоценка модели много раз
* Filter - самый быстрый - статистическая работа с датафреймом
* Lasso - чуть медленнее - одно обучение

* SHAP самый лучший
* Permutation чуть хуже
* Lasso ещё хуже
* Filter correlation ещё хуже

метод	стабильность
* Correlation	средняя - игнорирует взаимодействия
* Lasso	- низкая - может выкинуть разные фичи, зависит от масштаба, линейных зависимостей
* Permutation - средняя - зависит от случайности
* SHAP	- высокая - усредняет вклад

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import ElasticNet
import numpy as np

In [14]:
X_train, X_val, X_test, y_train, y_val, y_test = my_train_val_test_split(X, y, val_size=0.2)

In [15]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [18]:
from sklearn.linear_model import ElasticNet

def my_grid_search(X_train, y_train, X_val, y_val):
    alphas = [0.0001, 0.001, 0.01, 0.1, 1.0]
    l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

    best_score = float('inf')
    best_params = None

    for alpha in alphas:
        for l1 in l1_ratios:
            model = ElasticNet(alpha=alpha, l1_ratio=l1, random_state=21)
            model.fit(X_train, y_train)

            pred = model.predict(X_val)
            score = root_mean_squared_error(y_val, pred)

            if score < best_score:
                best_score = score
                best_params = (alpha, l1)

    return best_params, best_score

In [19]:
best_params_grid, best_score_grid = my_grid_search(
    X_train_s, y_train,
    X_val_s, y_val
)

print("Best (Grid):", best_params_grid, best_score_grid)

Best (Grid): (1.0, 0.7) 2159.345372387859


In [20]:
import numpy as np

def my_random_search(X_train, y_train, X_val, y_val, n_iter=20):
    best_score = float('inf')
    best_params = None

    rng = np.random.RandomState(21)

    for _ in range(n_iter):
        alpha = 10 ** rng.uniform(-4, 1)
        l1 = rng.uniform(0, 1)

        model = ElasticNet(alpha=alpha, l1_ratio=l1, random_state=21)
        model.fit(X_train, y_train)

        pred = model.predict(X_val)
        score = root_mean_squared_error(y_val, pred)

        if score < best_score:
            best_score = score
            best_params = (alpha, l1)

    return best_params, best_score

In [21]:
best_params_rand, best_score_rand = my_random_search(
    X_train_s, y_train,
    X_val_s, y_val,
    n_iter=30
)

print("Best (Random):", best_params_rand, best_score_rand)

Best (Random): (0.625911259201679, 0.3842500320901294) 2157.4730295049994


In [22]:
X_train_s = np.vstack([X_train_s, X_val_s])
y_train = np.concatenate([y_train, y_val])

In [23]:
best_alpha, best_l1 = best_params_rand

model = ElasticNet(
    alpha=best_alpha,
    l1_ratio=best_l1,
    random_state=21
)

model.fit(X_train_s, y_train)

ElasticNet(alpha=0.625911259201679, l1_ratio=0.3842500320901294,
           random_state=21)

In [27]:
test_pred = model.predict(X_test_s)
root_mean_squared_error(y_test, test_pred)

1922.6014242679596

Optuna

In [29]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.6 MB/s eta 0:00:00


In [32]:
import optuna

def objective(trial):
  alpha = trial.suggest_float('alpha', 1e-4, 10, log=True)
  l1_ratio = trial.suggest_float('l1_ratio', 0, 1)

  model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=21)
  model.fit(X_train_s, y_train)

  val_pred = model.predict(X_val_s)
  score = root_mean_squared_error(y_val, val_pred)

  return score


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

[I 2026-04-26 12:59:42,498] A new study created in memory with name: no-name-044df57f-e2d4-40ac-97a5-6e1b419ece58
[I 2026-04-26 12:59:42,729] Trial 0 finished with value: 2157.198682838696 and parameters: {'alpha': 0.0012425037245250765, 'l1_ratio': 0.2258230136150663}. Best is trial 0 with value: 2157.198682838696.
[I 2026-04-26 12:59:42,794] Trial 1 finished with value: 2141.816607719042 and parameters: {'alpha': 0.3324825546498377, 'l1_ratio': 0.841510870378113}. Best is trial 1 with value: 2141.816607719042.
[I 2026-04-26 12:59:42,839] Trial 2 finished with value: 2123.1129113169577 and parameters: {'alpha': 0.6715945339737366, 'l1_ratio': 0.5788513190993043}. Best is trial 2 with value: 2123.1129113169577.
[I 2026-04-26 12:59:43,021] Trial 3 finished with value: 2157.5422408562154 and parameters: {'alpha': 0.00013517118076737452, 'l1_ratio': 0.6513021702744383}. Best is trial 2 with value: 2123.1129113169577.
[I 2026-04-26 12:59:43,139] Trial 4 finished with value: 2154.9298433807

In [35]:
print(study.best_params)
print(study.best_value)

{'alpha': 0.5122695234209033, 'l1_ratio': 0.4720374371207574}
2123.0787765202226


In [36]:
best_alpha = study.best_params["alpha"]
best_l1 = study.best_params["l1_ratio"]

final_model = ElasticNet(
    alpha=best_alpha,
    l1_ratio=best_l1,
    random_state=21
)

X_full = np.vstack([X_train_s, X_val_s])
y_full = np.concatenate([y_train, y_val])

final_model.fit(X_full, y_full)

ElasticNet(alpha=0.5122695234209033, l1_ratio=0.4720374371207574,
           random_state=21)

In [38]:
test_pred = final_model.predict(X_test_s)

print("Optuna Test RMSE:", root_mean_squared_error(y_test, test_pred))

Optuna Test RMSE: 1893.3209642417394


CV

In [44]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=21)

def objective(trial):
  alpha = trial.suggest_float('alpha', 1e-4, 10, log=True)
  l1_ratio = trial.suggest_float('l1_ratio', 0, 1)

  scores = []
  for (train_idx, val_idx) in kf.split(X_train_s):
    X_tr, X_val = X_train_s[train_idx], X_train_s[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

  model = ElasticNet(
      alpha=alpha,
      l1_ratio=l1_ratio,
      random_state=21
  )

  model.fit(X_tr, y_tr)
  pred = model.predict(X_val)

  scores.append(root_mean_squared_error(y_val, pred))

  return np.mean(scores)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

[I 2026-04-26 13:18:44,221] A new study created in memory with name: no-name-f8daf665-77e1-4927-aadf-87caf0e9439f
[I 2026-04-26 13:18:44,334] Trial 0 finished with value: 52133.56313195221 and parameters: {'alpha': 0.024170420706725915, 'l1_ratio': 0.8525603270896528}. Best is trial 0 with value: 52133.56313195221.
[I 2026-04-26 13:18:44,477] Trial 1 finished with value: 52133.50469692993 and parameters: {'alpha': 0.0021604254737433346, 'l1_ratio': 0.730415443257444}. Best is trial 1 with value: 52133.50469692993.
[I 2026-04-26 13:18:44,637] Trial 2 finished with value: 52133.49591355299 and parameters: {'alpha': 0.0002474772866250176, 'l1_ratio': 0.4838631540295222}. Best is trial 2 with value: 52133.49591355299.
[I 2026-04-26 13:18:44,772] Trial 3 finished with value: 52133.501701803216 and parameters: {'alpha': 0.002459437340486467, 'l1_ratio': 0.8285803032173196}. Best is trial 2 with value: 52133.49591355299.
[I 2026-04-26 13:18:44,822] Trial 4 finished with value: 52155.692560470

In [45]:
print(study.best_params)
print(study.best_value)

best_alpha = study.best_params["alpha"]
best_l1 = study.best_params["l1_ratio"]

final_model = ElasticNet(
    alpha=best_alpha,
    l1_ratio=best_l1,
    random_state=21
)

X_full = np.vstack([X_train_s, X_val_s])
y_full = np.concatenate([y_train, y_val])

final_model.fit(X_full, y_full)

test_pred = final_model.predict(X_test_s)

print("Optuna Test RMSE:", root_mean_squared_error(y_test, test_pred))

{'alpha': 0.000249820336614239, 'l1_ratio': 0.9936905567861508}
52133.493535334084
Optuna Test RMSE: 1888.6940685598631
